# MAINTiDOC — Semaine 2 (suite) : mesure APRÈS correctifs, avec BGE-M3

Les **premiers passages** (avant correctifs) sont déjà enregistrés : `docs/eval_bge-m3_premier_passage.md` (jeu de 70) et
`docs/eval_150_bge-m3_premier_passage.md` (jeu figé de 150). Ce notebook mesure l'effet des correctifs décrits dans
`docs/eval_150_corrections.md`, et fait **la première mesure** du jeu de contrôle `holdout_60` (jamais lu avant).

Avant de lancer : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Tout exécuter*. Compte 10 à 15 minutes.

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],capture_output=True,text=True).stdout or "PAS DE GPU")
print("CUDA disponible :", torch.cuda.is_available())

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-fra > /dev/null
!pip -q install pymupdf qdrant-client pydantic pydantic-settings pytesseract reportlab pillow sentence-transformers pyyaml

In [ ]:
import os
if os.path.exists("MaintiDoc"):
    !rm -rf MaintiDoc
!git clone -q https://github.com/ziadserray04/MaintiDoc.git
%cd MaintiDoc
!git log --oneline | head -3
!python -m scripts.generate_corpus

In [ ]:
# Les jeux n'ont pas changé depuis leur gel (empreintes)
!sha256sum data/evaluation/eval_150.jsonl data/evaluation/holdout_60.jsonl && cat data/evaluation/eval_150.sha256 data/evaluation/holdout_60.sha256

In [ ]:
# BGE-M3 : jeu de 150 (80 exécutables) puis jeu de contrôle puis jeu de 70
!EMBEDDING_BACKEND=bge-m3 python -m scripts.eval_text_reference docs/eval_150_bge-m3_apres_correctifs.md data/evaluation/eval_150.jsonl | head -30
!EMBEDDING_BACKEND=bge-m3 python -m scripts.eval_text_reference docs/holdout_60_bge-m3_apres_correctifs.md data/evaluation/holdout_60.jsonl | head -30
!EMBEDDING_BACKEND=bge-m3 python -m scripts.eval_text_reference docs/eval_bge-m3_apres_correctifs.md | head -30
!EMBEDDING_BACKEND=bge-m3 python -m scripts.eval_smoke

In [ ]:
# Backend de test, mêmes jeux, pour comparaison
!EMBEDDING_BACKEND=hashing python -m scripts.eval_text_reference docs/eval_150_hashing_apres_correctifs.md data/evaluation/eval_150.jsonl > /dev/null
!EMBEDDING_BACKEND=hashing python -m scripts.eval_text_reference docs/holdout_60_hashing_apres_correctifs.md data/evaluation/holdout_60.jsonl > /dev/null

In [ ]:
# Tableau avant / après (réponses fausses = « answered » alors que ce n'était pas attendu ; abstentions à tort)
import re, os, pandas as pd
def stats(p):
    if not os.path.exists(p): return None
    t=open(p,encoding="utf-8").read(); f=re.findall(r"^- (\w+) \((\w+)\) .*? -> (\w+) :",t,re.M)
    g=lambda k:(re.search(rf"\|\s*{re.escape(k)}[^|]*\|\s*([^|]+)\|",t) or [None,"?"])[1].strip()
    tot=re.search(r"Total\*\*\s*\|\s*\*\*(\d+/\d+)",t)
    return {"Total":tot[1] if tot else "?","Recall@5":g("Recall@5"),"Citations":g("Réponses avec bonne citation"),
            "Fausses réponses":sum(st=="answered" for _,_,st in f),"Abstentions à tort":sum(st=="abstained" for _,_,st in f)}
rows={"150 avant":"docs/eval_150_bge-m3_premier_passage.md","150 après":"docs/eval_150_bge-m3_apres_correctifs.md",
      "contrôle (1re mesure)":"docs/holdout_60_bge-m3_apres_correctifs.md","70 avant":"docs/eval_bge-m3_premier_passage.md","70 après":"docs/eval_bge-m3_apres_correctifs.md"}
print(pd.DataFrame({k:stats(v) for k,v in rows.items() if stats(v)}).T)

In [ ]:
!zip -q resultats_apres_correctifs.zip docs/eval_150_bge-m3_apres_correctifs.md docs/holdout_60_bge-m3_apres_correctifs.md docs/eval_bge-m3_apres_correctifs.md docs/eval_150_hashing_apres_correctifs.md docs/holdout_60_hashing_apres_correctifs.md
from google.colab import files
files.download("resultats_apres_correctifs.zip")